# Module 3 — Time-Series Resampling: OHLC Data for Stocks & Crypto

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Intermediate  
> **Runtime**: ~3 minutes  
> **Key Libraries**: Pandas, NumPy, Matplotlib, Plotly, Statsmodels  
> **Prerequisite**: Module 2 (GroupBy & Aggregation)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Tick Data Generation](#3-synthetic-tick-data-generation)
4. [Resampling to OHLC Candles](#4-resampling-to-ohlc-candles)
5. [Technical Indicators](#5-technical-indicators)
   - 5a. Moving Averages (SMA & EMA)
   - 5b. Bollinger Bands
   - 5c. RSI (Relative Strength Index)
   - 5d. MACD
6. [Multi-Timeframe Analysis](#6-multi-timeframe-analysis)
7. [Volume Analysis](#7-volume-analysis)
8. [Interactive Candlestick Dashboard](#8-interactive-candlestick-dashboard)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why does time-series resampling matter at Revolut?

Revolut offers **crypto trading** (BTC, ETH, SOL) and **commodity exposure** (Gold, Silver)
to 30M+ users. The data-science team needs OHLC (Open-High-Low-Close) aggregations for:

| Use Case | Team | Why It Matters |
|----------|------|----------------|
| **Real-time price feeds** | Trading | Detect flash crashes and circuit-breaker triggers |
| **Volatility monitoring** | Risk | Adjust margin requirements when BTC volatility spikes |
| **User-behavior analysis** | Product | Correlate price spikes with buy/sell volume surges |
| **Technical indicators** | Quant | RSI/MACD signals for automated trading strategies |

**The core skill**: converting **irregular tick data** (thousands of trades per minute)
into **regular OHLC candles** (1m, 5m, 1h, 1d) using Pandas `.resample()`.

In this notebook we will:
1. Generate 2M synthetic crypto ticks over 30 days.
2. Resample to 1-minute, 5-minute, and hourly OHLC candles.
3. Compute SMA, EMA, Bollinger Bands, RSI, and MACD.
4. Build an interactive Plotly candlestick dashboard.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="darkgrid", palette="muted")

print("✓ All imports loaded")
print(f"  Pandas {pd.__version__}  |  NumPy {np.__version__}")

---
## §3 · Synthetic Tick Data Generation

We simulate **2 million crypto trades** (BTC/USD) over 30 days with:
- **Geometric Brownian Motion** for realistic price paths
- **Intraday seasonality** (higher volume during London/NY hours)
- **Random volatility spikes** (simulating news events)

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_TICKS = 2_000_000
START   = pd.Timestamp("2025-05-01")
END     = pd.Timestamp("2025-05-31")
SPAN_S  = (END - START).total_seconds()

INITIAL_PRICE = 65_000.0   # BTC/USD starting price
DRIFT         = 0.0001     # slight upward bias
VOLATILITY    = 0.002      # per-tick volatility

print(f"Generating {N_TICKS:,} synthetic BTC/USD ticks over 30 days …")

# ── Timestamps with intraday seasonality ─────────────────────────
# Simulate higher activity during London (08-16 UTC) and NY (13-21 UTC)
raw_seconds = np.random.uniform(0, SPAN_S, N_TICKS)
raw_ts = START + pd.to_timedelta(raw_seconds, unit="s")

# Weight: higher probability during trading hours
hours = raw_ts.hour + raw_ts.minute / 60
weights = np.where((hours >= 8) & (hours <= 21), 2.5, 1.0)
weights /= weights.sum()

# Resample timestamps according to weights
sampled_idx = np.random.choice(N_TICKS, size=N_TICKS, replace=True, p=weights)
timestamps = raw_ts.iloc[sampled_idx].sort_values().reset_index(drop=True)

# ── Price simulation (Geometric Brownian Motion) ─────────────────
log_returns = np.random.normal(DRIFT, VOLATILITY, N_TICKS)

# Inject 50 random "news events" with large moves
news_events = np.random.choice(N_TICKS, size=50, replace=False)
log_returns[news_events] += np.random.choice([-0.05, 0.05], size=50)

# Cumulative sum → price path
cum_returns = np.cumsum(log_returns)
prices = INITIAL_PRICE * np.exp(cum_returns)

# ── Volume simulation ────────────────────────────────────────────
# Higher volume during volatile periods
vol_base = np.random.lognormal(mean=2, sigma=1.0, size=N_TICKS)
vol_spike = np.abs(log_returns) * 5000   # more volume when price moves a lot
volumes = np.round(vol_base + vol_spike, 4)

# ── Assemble DataFrame ───────────────────────────────────────────
ticks = pd.DataFrame({
    "timestamp": timestamps,
    "price":     np.round(prices, 2),
    "volume":    volumes,
})
ticks.set_index("timestamp", inplace=True)

print(f"✓ Shape: {ticks.shape}")
print(f"  Price range: ${ticks['price'].min():,.2f} – ${ticks['price'].max():,.2f}")
ticks.head(10)

In [ ]:
# Quick look at the raw price path
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ticks.index, y=ticks["price"],
    mode="lines", name="BTC/USD Tick Price",
    line=dict(color="#636EFA", width=0.8),
))
fig.update_layout(
    title="Raw Tick Data: BTC/USD (2M ticks over 30 days)",
    xaxis_title="Date", yaxis_title="Price (USD)",
    height=420,
)
fig.show()

---
## §4 · Resampling to OHLC Candles

### 4.1 — 1-Minute Candles

The `.resample()` method groups irregular ticks into regular time buckets.

In [ ]:
# Resample to 1-minute OHLC
ohlc_1m = ticks["price"].resample("1min").ohlc()
ohlc_1m["volume"] = ticks["volume"].resample("1min").sum()
ohlc_1m.dropna(inplace=True)

print(f"✓ 1-minute candles: {len(ohlc_1m):,}")
print(f"  Date range: {ohlc_1m.index.min()} → {ohlc_1m.index.max()}")
ohlc_1m.head()

### 4.2 — 5-Minute and Hourly Candles

In [ ]:
# 5-minute candles
ohlc_5m = ticks["price"].resample("5min").ohlc()
ohlc_5m["volume"] = ticks["volume"].resample("5min").sum()
ohlc_5m.dropna(inplace=True)

# 1-hour candles
ohlc_1h = ticks["price"].resample("1h").ohlc()
ohlc_1h["volume"] = ticks["volume"].resample("1h").sum()
ohlc_1h.dropna(inplace=True)

# 1-day candles
ohlc_1d = ticks["price"].resample("1D").ohlc()
ohlc_1d["volume"] = ticks["volume"].resample("1D").sum()
ohlc_1d.dropna(inplace=True)

print(f"5-min candles : {len(ohlc_5m):,}")
print(f"1-hour candles: {len(ohlc_1h):,}")
print(f"1-day candles : {len(ohlc_1d):,}")

ohlc_1d

### 4.3 — Understanding `.ohlc()` Aggregation

| Candle Field | Aggregation |
|--------------|-------------|
| **Open** | First price in the bucket |
| **High** | Maximum price in the bucket |
| **Low** | Minimum price in the bucket |
| **Close** | Last price in the bucket |
| **Volume** | Sum of all trade volumes in the bucket |

In [ ]:
# Demonstrate manually for one 5-minute window
sample_window = ticks["2025-05-01 12:00":"2025-05-01 12:05"]
print(f"Ticks in 12:00–12:05 window: {len(sample_window)}")
print(f"  Open  (first): ${sample_window['price'].iloc[0]:,.2f}")
print(f"  High  (max)  : ${sample_window['price'].max():,.2f}")
print(f"  Low   (min)  : ${sample_window['price'].min():,.2f}")
print(f"  Close (last) : ${sample_window['price'].iloc[-1]:,.2f}")
print(f"  Volume (sum) : {sample_window['volume'].sum():,.2f}")
print(f"\nMatches resampled candle:")
ohlc_5m.loc["2025-05-01 12:00"]

---
## §5 · Technical Indicators

### 5a — Simple Moving Average (SMA) & Exponential Moving Average (EMA)

In [ ]:
# Work with hourly candles for cleaner visualization
df_h = ohlc_1h.copy()

# SMA: arithmetic mean of last N periods
df_h["SMA_20"] = df_h["close"].rolling(20).mean()
df_h["SMA_50"] = df_h["close"].rolling(50).mean()

# EMA: weighted average (more weight on recent prices)
df_h["EMA_20"] = df_h["close"].ewm(span=20, adjust=False).mean()
df_h["EMA_50"] = df_h["close"].ewm(span=50, adjust=False).mean()

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["close"], name="Close",
                         line=dict(color="gray", width=1)))
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["SMA_20"], name="SMA 20",
                         line=dict(color="#636EFA", width=2)))
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["EMA_20"], name="EMA 20",
                         line=dict(color="#EF553B", width=2, dash="dash")))

fig.update_layout(title="SMA vs EMA (20-period) on Hourly BTC/USD",
                  yaxis_title="Price (USD)", height=420,
                  legend=dict(orientation="h", y=1.12))
fig.show()

print("💡 EMA reacts faster to price changes than SMA (less lag)")

### 5b — Bollinger Bands

Bollinger Bands = SMA ± 2σ. Price touching the upper band suggests overbought;
touching the lower band suggests oversold.

In [ ]:
# Bollinger Bands (20-period SMA ± 2 standard deviations)
df_h["BB_mid"]   = df_h["close"].rolling(20).mean()
df_h["BB_std"]   = df_h["close"].rolling(20).std()
df_h["BB_upper"] = df_h["BB_mid"] + 2 * df_h["BB_std"]
df_h["BB_lower"] = df_h["BB_mid"] - 2 * df_h["BB_std"]
df_h["BB_width"] = (df_h["BB_upper"] - df_h["BB_lower"]) / df_h["BB_mid"] * 100  # % width

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["close"], name="Close",
                         line=dict(color="white", width=1.5)))
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["BB_upper"], name="Upper Band",
                         line=dict(color="rgba(173,216,230,0.5)", width=1)))
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["BB_lower"], name="Lower Band",
                         fill="tonexty", fillcolor="rgba(173,216,230,0.1)",
                         line=dict(color="rgba(173,216,230,0.5)", width=1)))
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["BB_mid"], name="SMA 20",
                         line=dict(color="#FFD700", width=1.5, dash="dot")))

fig.update_layout(
    title="Bollinger Bands (20, 2σ) on Hourly BTC/USD",
    yaxis_title="Price (USD)", height=450,
    plot_bgcolor="rgb(20,20,30)", paper_bgcolor="rgb(20,20,30)",
    font_color="white",
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print(f"Average BB Width: {df_h['BB_width'].mean():.2f}%")
print(f"  Narrow bands (width < 2%) indicate low volatility — a breakout may follow")

### 5c — RSI (Relative Strength Index)

RSI measures momentum: RSI > 70 = overbought, RSI < 30 = oversold.

In [ ]:
def compute_rsi(series, period=14):
    """Compute RSI using Wilder's smoothing method."""
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    
    # Wilder's smoothing (equivalent to EWM with alpha=1/period)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df_h["RSI_14"] = compute_rsi(df_h["close"], period=14)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["RSI_14"], name="RSI 14",
                         line=dict(color="#00CC96", width=2)))
fig.add_hline(y=70, line_dash="dash", line_color="red",
              annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="green",
              annotation_text="Oversold (30)")
fig.add_hline(y=50, line_dash="dot", line_color="gray")

fig.update_layout(
    title="RSI (14-period) on Hourly BTC/USD",
    yaxis_title="RSI", height=350,
    yaxis_range=[0, 100],
    plot_bgcolor="rgb(20,20,30)", paper_bgcolor="rgb(20,20,30)",
    font_color="white",
)
fig.show()

# Stats
print(f"RSI Statistics:")
print(f"  Mean:   {df_h['RSI_14'].mean():.1f}")
print(f"  Min:    {df_h['RSI_14'].min():.1f}")
print(f"  Max:    {df_h['RSI_14'].max():.1f}")
print(f"  % time overbought (>70): {(df_h['RSI_14'] > 70).mean() * 100:.1f}%")
print(f"  % time oversold  (<30): {(df_h['RSI_14'] < 30).mean() * 100:.1f}%")

### 5d — MACD (Moving Average Convergence Divergence)

MACD = EMA(12) − EMA(26). The signal line is EMA(9) of MACD.

In [ ]:
# MACD components
df_h["EMA_12"] = df_h["close"].ewm(span=12, adjust=False).mean()
df_h["EMA_26"] = df_h["close"].ewm(span=26, adjust=False).mean()
df_h["MACD"]        = df_h["EMA_12"] - df_h["EMA_26"]
df_h["MACD_signal"] = df_h["MACD"].ewm(span=9, adjust=False).mean()
df_h["MACD_hist"]   = df_h["MACD"] - df_h["MACD_signal"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("BTC/USD Price", "MACD"),
                    row_heights=[0.6, 0.4], vertical_spacing=0.08)

# Price
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["close"], name="Close",
                         line=dict(color="white", width=1.5)), row=1, col=1)

# MACD line
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["MACD"], name="MACD",
                         line=dict(color="#636EFA", width=2)), row=2, col=1)
# Signal line
fig.add_trace(go.Scatter(x=df_h.index, y=df_h["MACD_signal"], name="Signal",
                         line=dict(color="#EF553B", width=2, dash="dash")), row=2, col=1)
# Histogram
colors = ["#00CC96" if v >= 0 else "#EF553B" for v in df_h["MACD_hist"]]
fig.add_trace(go.Bar(x=df_h.index, y=df_h["MACD_hist"], name="Histogram",
                     marker_color=colors, opacity=0.6), row=2, col=1)

fig.update_layout(
    height=550,
    plot_bgcolor="rgb(20,20,30)", paper_bgcolor="rgb(20,20,30)",
    font_color="white",
    legend=dict(orientation="h", y=1.08),
)
fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
fig.update_yaxes(title_text="MACD", row=2, col=1)
fig.show()

print("💡 MACD crossover (MACD crosses above signal) = bullish signal")
print("💡 MACD crossunder (MACD crosses below signal) = bearish signal")

---
## §6 · Multi-Timeframe Analysis

Comparing daily, hourly, and 5-minute views of the same data.

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=False,
                    subplot_titles=("Daily Candles (1D)", "Hourly Candles (1H)", "5-Minute Candles (5m)"),
                    row_heights=[0.4, 0.35, 0.25], vertical_spacing=0.08)

for i, (df_tf, name) in enumerate([
    (ohlc_1d, "1D"), (ohlc_1h.tail(500), "1H"), (ohlc_5m.tail(500), "5m")
], start=1):
    fig.add_trace(go.Candlestick(
        x=df_tf.index, open=df_tf["open"], high=df_tf["high"],
        low=df_tf["low"], close=df_tf["close"],
        name=name, increasing_line_color="#00CC96", decreasing_line_color="#EF553B",
    ), row=i, col=1)

fig.update_layout(height=700, showlegend=False,
                  plot_bgcolor="rgb(20,20,30)", paper_bgcolor="rgb(20,20,30)",
                  font_color="white")
fig.update_xaxes(rangeslider_visible=False)
fig.show()

---
## §7 · Volume Analysis

Volume confirms price trends: rising price + rising volume = strong trend.

In [ ]:
# Daily volume and price change
daily_analysis = ohlc_1d.copy()
daily_analysis["price_change_pct"] = (
    (daily_analysis["close"] - daily_analysis["open"]) / daily_analysis["open"] * 100
)
daily_analysis["volume_ma_7"] = daily_analysis["volume"].rolling(7).mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Daily Price Change %", "Daily Volume"),
                    row_heights=[0.5, 0.5], vertical_spacing=0.1)

# Price change bars
colors = ["#00CC96" if x >= 0 else "#EF553B" for x in daily_analysis["price_change_pct"]]
fig.add_trace(go.Bar(
    x=daily_analysis.index, y=daily_analysis["price_change_pct"],
    marker_color=colors, name="Daily Change %",
), row=1, col=1)

# Volume with MA
fig.add_trace(go.Bar(
    x=daily_analysis.index, y=daily_analysis["volume"],
    marker_color="rgba(100,149,237,0.6)", name="Volume",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily_analysis.index, y=daily_analysis["volume_ma_7"],
    name="7-day MA Volume", line=dict(color="#FFD700", width=2),
), row=2, col=1)

fig.update_layout(height=500, showlegend=False,
                  title="Daily Price Change & Volume Analysis")
fig.update_yaxes(title_text="Change %", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.show()

# Correlation
corr = daily_analysis[["price_change_pct", "volume"]].corr().iloc[0, 1]
print(f"Correlation (daily change vs volume): {corr:.3f}")
print("  → Positive correlation confirms volume-follows-price pattern")

---
## §8 · Interactive Candlestick Dashboard

A professional-grade trading chart with all indicators.

In [ ]:
# Use hourly data with all indicators
df_dash = df_h.copy()

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    subplot_titles=("BTC/USD — Hourly with Bollinger Bands", "Volume", "RSI (14)", "MACD"),
    row_heights=[0.45, 0.15, 0.2, 0.2],
    vertical_spacing=0.04,
)

# Candlestick
fig.add_trace(go.Candlestick(
    x=df_dash.index, open=df_dash["open"], high=df_dash["high"],
    low=df_dash["low"], close=df_dash["close"],
    name="OHLC", increasing_line_color="#00CC96", decreasing_line_color="#EF553B",
), row=1, col=1)

# Bollinger Bands
fig.add_trace(go.Scatter(x=df_dash.index, y=df_dash["BB_upper"], name="BB Upper",
                         line=dict(color="rgba(173,216,230,0.5)", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_dash.index, y=df_dash["BB_lower"], name="BB Lower",
                         fill="tonexty", fillcolor="rgba(173,216,230,0.1)",
                         line=dict(color="rgba(173,216,230,0.5)", width=1)), row=1, col=1)

# Volume
vol_colors = ["#00CC96" if c >= o else "#EF553B"
              for c, o in zip(df_dash["close"], df_dash["open"])]
fig.add_trace(go.Bar(x=df_dash.index, y=df_dash["volume"], name="Volume",
                     marker_color=vol_colors, opacity=0.7), row=2, col=1)

# RSI
fig.add_trace(go.Scatter(x=df_dash.index, y=df_dash["RSI_14"], name="RSI",
                         line=dict(color="#00CC96", width=1.5)), row=3, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=3, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=3, col=1)

# MACD
fig.add_trace(go.Scatter(x=df_dash.index, y=df_dash["MACD"], name="MACD",
                         line=dict(color="#636EFA", width=1.5)), row=4, col=1)
fig.add_trace(go.Scatter(x=df_dash.index, y=df_dash["MACD_signal"], name="Signal",
                         line=dict(color="#EF553B", width=1.5, dash="dash")), row=4, col=1)
macd_colors = ["#00CC96" if v >= 0 else "#EF553B" for v in df_dash["MACD_hist"]]
fig.add_trace(go.Bar(x=df_dash.index, y=df_dash["MACD_hist"], name="Hist",
                     marker_color=macd_colors, opacity=0.6), row=4, col=1)

fig.update_layout(
    height=850,
    plot_bgcolor="rgb(20,20,30)", paper_bgcolor="rgb(20,20,30)",
    font_color="white",
    title="📈 BTC/USD Trading Dashboard — May 2025",
    xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"),
)

# Hide weekends on x-axis (optional for stocks; crypto trades 24/7)
fig.update_xaxes(rangeslider_visible=False)

fig.show()

print("✅ Interactive dashboard complete")
print("   Hover over candles to see OHLC values")
print("   Use the range slider at the bottom to zoom into specific periods")

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for a Fintech Data Scientist
>
> | Technique | Use Case | Key Method |
> |-----------|----------|------------|
> | **`.resample("1min").ohlc()`** | Tick → candle conversion | Core aggregation for all time-series |
> | **SMA / EMA** | Trend detection, signal smoothing | `.rolling(N).mean()` / `.ewm(span=N)` |
> | **Bollinger Bands** | Volatility monitoring, breakout detection | SMA ± 2σ rolling window |
> | **RSI** | Overbought/oversold signals | Wilder's smoothing of gains/losses |
> | **MACD** | Momentum and trend-reversal signals | EMA(12) − EMA(26), signal = EMA(9) |
> | **Multi-timeframe** | Confirm signals across resolutions | Resample to 1m, 5m, 1h, 1D |
>
> ### 💡 Revolut-Specific Applications
>
> 1. **Crypto Trading Desk**: Use Bollinger Band width to adjust margin requirements dynamically.
>    - Narrow bands → increase margin (low vol, breakout imminent)
>    - Wide bands → relax margin (high vol, mean-reversion likely)
>
> 2. **Risk Monitoring**: Alert the risk team when RSI > 80 or RSI < 20 on hourly BTC/ETH.
>
> 3. **User Behavior**: Correlate RSI extremes with buy/sell order volume to predict liquidity needs.
>
> 4. **Automated Strategies**: MACD crossover signals can trigger automated rebalancing for "Smart Portfolios".
>
> ### 📌 Resampling Cheat Sheet
>
> ```python
> # Time-based resampling frequencies
> "1min"   # 1 minute
> "5min"   # 5 minutes
> "1h"     # 1 hour
> "1D"     # 1 day
> "1W"     # 1 week (Sunday-aligned)
> "1M"     # 1 month (calendar)
>
> # OHLC aggregation
> df["price"].resample("1h").ohlc()
>
> # Custom aggregation
> df.resample("1h").agg({
>     "price": ["first", "max", "min", "last"],
>     "volume": "sum"
> })
> ```